In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 인코더 LSTM 과 디코더 LSTM(Attention)로 번역하기
- attention 

## 1. 패키지 impiort & 하이퍼파라미터
- 하이퍼마라미터 : 모델의 정확도 및 학습속도에 영향을 미치는 변수

In [2]:
import numpy as np
import pandas as pd
from time import time

from tensorflow.keras.layers import Input,LSTM, Dense, Attention, Concatenate
from tensorflow.keras.models import Model
from  tensorflow.keras.utils import to_categorical

# 하이퍼파라미터
MY_HIDDEN = 128
MY_EPOCH = 500

## 2. 번역데이터 불러오기


In [4]:
raw = pd.read_csv('data/translate.csv', header=None)
eng_kor = raw.values.tolist() # 데이터프레임을 리스트로 변환
print('번역데이터 수 :', len(eng_kor))

번역데이터 수 : 110


## 3. 영어알파벳과 한글문자 리스트 만들기

In [5]:
e_alpha = [c for c in 'SEPabcdefghijklmnopqrstuvwexyz']
e_alpha
{c:i for i, c in enumerate(e_alpha)}
korean = ''.join([data[1] for data in eng_kor])
print(set([ch for ch in korean]))
k_ch = list(set([ch for ch in korean]))
k_ch.sort()


{'릎', '광', '날', '우', '바', '어', '단', '연', '한', '노', '뉴', '금', '비', '책', '험', '모', '동', '람', '나', '규', '랑', '뿌', '름', '간', '남', '파', '굴', '지', '구', '키', '무', '왼', '망', '복', '늦', '출', '여', '읽', '미', '서', '색', '많', '주', '휴', '먼', '약', '언', '붕', '메', '부', '이', '스', '머', '탈', '넓', '사', '류', '의', '방', '운', '적', '게', '음', '물', '칙', '다', '높', '수', '놀', '자', '아', '관', '내', '짜', '도', '익', '장', '행', '얇', '래', '얼', '깊', '을', '들', '팔', '것', '획', '좋', '오', '식', '핑', '고', '용', '크', '제', '합', '입', '생', '통', '명', '소', '찾', '가', '감', '쉽', '멍', '램', '움', '번', '유', '선', '피', '싸', '실', '쪽', '흐', '녀', '위', '개', '농', '요', '편', '그', '각', '옥', '반', '기', '매', '목', '택', '계', '손', '리', '회', '작', '거', '시', '해', '인', '은', '상', '분'}


In [6]:
k_alpha = pd.read_csv('data/korean.csv', header=None)[0].tolist()
k_alpha == k_ch  # 순서와 내용이 모두 같음

True

In [7]:
from collections import Counter
list1 = ['가', '간', '나']
list2 = ['간', '나', '가']
list1 == list2
Counter(list1) == Counter(list2)

True

In [8]:
alpha = e_alpha + k_alpha

alpha_total_size = len(alpha)


## 4. 문자당 num를 갖는 dict 만들기

In [9]:
char_to_num = {}
for i, c in enumerate(alpha):
    char_to_num[c] = i

In [10]:
char_to_num = {c : i for i, c in enumerate(alpha)}
print(char_to_num)

{'S': 0, 'E': 1, 'P': 2, 'a': 3, 'b': 4, 'c': 5, 'd': 6, 'e': 26, 'f': 8, 'g': 9, 'h': 10, 'i': 11, 'j': 12, 'k': 13, 'l': 14, 'm': 15, 'n': 16, 'o': 17, 'p': 18, 'q': 19, 'r': 20, 's': 21, 't': 22, 'u': 23, 'v': 24, 'w': 25, 'x': 27, 'y': 28, 'z': 29, '가': 30, '각': 31, '간': 32, '감': 33, '개': 34, '거': 35, '것': 36, '게': 37, '계': 38, '고': 39, '관': 40, '광': 41, '구': 42, '굴': 43, '규': 44, '그': 45, '금': 46, '기': 47, '깊': 48, '나': 49, '날': 50, '남': 51, '내': 52, '넓': 53, '녀': 54, '노': 55, '놀': 56, '농': 57, '높': 58, '뉴': 59, '늦': 60, '다': 61, '단': 62, '도': 63, '동': 64, '들': 65, '람': 66, '랑': 67, '래': 68, '램': 69, '류': 70, '름': 71, '릎': 72, '리': 73, '많': 74, '망': 75, '매': 76, '머': 77, '먼': 78, '멍': 79, '메': 80, '명': 81, '모': 82, '목': 83, '무': 84, '물': 85, '미': 86, '바': 87, '반': 88, '방': 89, '번': 90, '복': 91, '부': 92, '분': 93, '붕': 94, '비': 95, '뿌': 96, '사': 97, '상': 98, '색': 99, '생': 100, '서': 101, '선': 102, '소': 103, '손': 104, '수': 105, '쉽': 106, '스': 107, '시': 108, '식': 109, '실': 110, '싸': 11

In [11]:
data= eng_kor[0]
print(data)
print(char_to_num['c'],char_to_num['o'],char_to_num['l'],char_to_num['d'])
print('인코더 입력(원핫인코딩전) ',[char_to_num[c] for c in data[0]])
print('디코더 입력(원핫인코딩전) : ',[char_to_num[c] for c in 'S'+data[1]])
print('디코더 출력(원핫인코딩x)', [char_to_num[c] for c in data[1]+'E'])

['cold', '감기']
5 17 14 6
인코더 입력(원핫인코딩전)  [5, 17, 14, 6]
디코더 입력(원핫인코딩전) :  [0, 33, 47]
디코더 출력(원핫인코딩x) [33, 47, 1]


In [12]:
# 희소행렬의 원핫인코딩방법1  주의 get dummy 를 쓰면 안됨

In [13]:
to_categorical([5,7,6,8], num_classes=10)

array([[0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]], dtype=float32)

In [14]:
# 희소 행렬의 원핫인코딩방법2
np.eye(10)

array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]])

In [15]:
def encoding(eng_kor=eng_kor):
    enc_in = [] # 인코더 입력
    dec_in = [] # 디코더 입력
    dec_out = [] # 디코더 출력(타겟)
    for data in eng_kor:
        # 인코더 입력데이터(영어알파벳 -> 숫자 -> 원핫인코딩)
        eng = [char_to_num[c] for c in data[0]]
        eng_one = np.eye(alpha_total_size)[eng]
        # print('영어 :', eng, eng_one)
        enc_in.append(eng_one) # eng_one의 shape : (4,171)
        
        # 디코더 입력데이터("S한글" -> 숫자 -> 원핫인코딩)
        kor = [char_to_num[c] for c in "S"+data[1]]
        #kor_one = to_categorical(kor, num_classes=alpha_total_size)
        kor_one = np.eye(alpha_total_size)[kor] # kor_one의 shape : (3, 171)
        # print('한글 :', kor, kor_one)
        dec_in.append(kor_one)
        
        # 디코더 출력데이터("한글E" -> 숫자)
        kor = [char_to_num[c] for c in data[1]+"E"]
        # print(kor)
        dec_out.append(kor)
    return enc_in, dec_in, dec_out

In [17]:
sample = [['cold', '감기'], ['wood','나무']]
x_enc, x_dec, y_dec = encoding(sample)
X_enc = np.array(x_enc)
X_dec = np.array(x_dec)
Y_dec = np.array(y_dec)
X_enc.shape, X_dec.shape, Y_dec.shape # (2,3,1)

((2, 4, 172), (2, 3, 172), (2, 3))

# 6. 전체 입력데이터, 타겟데이터 준비

In [18]:
x_enc, x_dec, y_dec = encoding(eng_kor)
X_enc = np.array(x_enc)
X_dec = np.array(x_dec)
Y_dec = np.expand_dims(y_dec, axis=-1)
#Y_dec = np.array(y_dec[..., np.newaxis])
X_enc.shape, X_dec.shape, Y_dec.shape

((110, 4, 172), (110, 3, 172), (110, 3, 1))

## 7. 모델 구현

In [21]:
# 인코더 LSTM
ENC_IN = Input(shape=(4, alpha_total_size)) # alpha_total_size:171

_, state_h, state_c = LSTM(units=MY_HIDDEN, # MY_HIDDEN:128
                           return_state=True, # return_state=True h값과 c 받기
                           # return_sequences=False # LSTM윗 출력 안 받음
                          )(ENC_IN) 

# 인코더와 디코더 연결 고리
LINK = [state_h, state_c]

# 디코더 LSTM
DEC_IN = Input(shape=(3, alpha_total_size))
DEC_MID = LSTM(units=MY_HIDDEN, # 128
              # return_state=False,
              return_sequences=True, # 윗 출력 받음
              )(DEC_IN,
               initial_state=LINK)

# 최종 출력층
DEC_OUT = Dense(units=alpha_total_size,
               activation='softmax')(DEC_MID)
# 모델
model = Model(inputs=[ENC_IN, DEC_IN],
             outputs=DEC_OUT)
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_4 (InputLayer)           [(None, 4, 172)]     0           []                               
                                                                                                  
 input_5 (InputLayer)           [(None, 3, 172)]     0           []                               
                                                                                                  
 lstm_2 (LSTM)                  [(None, 128),        154112      ['input_4[0][0]']                
                                 (None, 128),                                                     
                                 (None, 128)]                                                     
                                                                                            

## 7. 모델구현

In [24]:
# 인코더입력
ENC_IN = Input(shape=(4, alpha_total_size)) # alpha_total_size:171

# 인코드 LSTM : 모든 LSTM 스텝 출력
ENC_OUT, state_h, state_c = LSTM(units=MY_HIDDEN,
    return_sequences=True, # LSTM 모든 스텝의 윗 출력
    return_state=True #  h값과 c값 받기
    )(ENC_IN)

# 디코더 입력
DEC_IN = Input(shape=(3, alpha_total_size))

# 디코더 LSTM
DEC_LSTM_OUT, _, _ = LSTM(units=MY_HIDDEN, # 128
                    return_sequences=True,
                    return_state=True)(DEC_IN,
                                      initial_state=[state_h,
                                                    state_c])

# Attention vector(매커니즘 정의)
CONTEXT_VECTOR = Attention()([DEC_LSTM_OUT, ENC_OUT])

# 켄텍스트벡터와 디코더LSTM 결과를 concat
CONTEXT_AND_LSTM_OUT = Concatenate()([CONTEXT_VECTOR, 
                                      DEC_LSTM_OUT])
# 출력층
OUT = Dense(units=alpha_total_size,
           activation='softmax')(CONTEXT_AND_LSTM_OUT)

# 모델 정의
model = Model(inputs=[ENC_IN, DEC_IN], outputs=OUT)
model

In [25]:
model.compile(loss='sparse_categorical_crossentropy',
             optimizer='rmsprop',
             metrics=['accuracy'] # loss만 로그 출력
             )
begin = time()
model.fit([X_enc, X_dec], Y_dec,
         epochs=MY_EPOCH,
         verbose=1)
end = time()
print('학습시간 :', end-begin)

Epoch 1/500
4/4 [==============================] - 3s 13ms/step - loss: 5.1207 - accuracy: 0.2152
Epoch 2/500
4/4 [==============================] - 0s 13ms/step - loss: 4.9734 - accuracy: 0.3333
Epoch 3/500
4/4 [==============================] - 0s 13ms/step - loss: 4.3542 - accuracy: 0.3333
Epoch 4/500
4/4 [==============================] - 0s 13ms/step - loss: 3.4576 - accuracy: 0.3333
Epoch 5/500
4/4 [==============================] - 0s 12ms/step - loss: 3.3969 - accuracy: 0.3333
Epoch 6/500
4/4 [==============================] - 0s 12ms/step - loss: 3.3537 - accuracy: 0.3333
Epoch 7/500
4/4 [==============================] - 0s 14ms/step - loss: 3.3194 - accuracy: 0.3333
Epoch 8/500
4/4 [==============================] - 0s 14ms/step - loss: 3.2892 - accuracy: 0.3333
Epoch 9/500
4/4 [==============================] - 0s 19ms/step - loss: 3.2613 - accuracy: 0.3333
Epoch 10/500
4/4 [==============================] - 0s 13ms/step - loss: 3.2365 - accuracy: 0.3364
Epoch 11/500
4/4 [=

4/4 [==============================] - 0s 12ms/step - loss: 0.5546 - accuracy: 0.9576
Epoch 84/500
4/4 [==============================] - 0s 12ms/step - loss: 0.5290 - accuracy: 0.9515
Epoch 85/500
4/4 [==============================] - 0s 10ms/step - loss: 0.5093 - accuracy: 0.9636
Epoch 86/500
4/4 [==============================] - 0s 11ms/step - loss: 0.4870 - accuracy: 0.9545
Epoch 87/500
4/4 [==============================] - 0s 11ms/step - loss: 0.4678 - accuracy: 0.9667
Epoch 88/500
4/4 [==============================] - 0s 18ms/step - loss: 0.4634 - accuracy: 0.9667
Epoch 89/500
4/4 [==============================] - 0s 18ms/step - loss: 0.4307 - accuracy: 0.9697
Epoch 90/500
4/4 [==============================] - 0s 12ms/step - loss: 0.4180 - accuracy: 0.9697
Epoch 91/500
4/4 [==============================] - 0s 10ms/step - loss: 0.3868 - accuracy: 0.9697
Epoch 92/500
4/4 [==============================] - 0s 11ms/step - loss: 0.3754 - accuracy: 0.9697
Epoch 93/500
4/4 [=====

4/4 [==============================] - 0s 12ms/step - loss: 0.0115 - accuracy: 1.0000
Epoch 166/500
4/4 [==============================] - 0s 12ms/step - loss: 0.0161 - accuracy: 0.9970
Epoch 167/500
4/4 [==============================] - 0s 18ms/step - loss: 0.0151 - accuracy: 0.9970
Epoch 168/500
4/4 [==============================] - 0s 11ms/step - loss: 0.0084 - accuracy: 1.0000
Epoch 169/500
4/4 [==============================] - 0s 14ms/step - loss: 0.0088 - accuracy: 1.0000
Epoch 170/500
4/4 [==============================] - 0s 13ms/step - loss: 0.0116 - accuracy: 1.0000
Epoch 171/500
4/4 [==============================] - 0s 13ms/step - loss: 0.0085 - accuracy: 1.0000
Epoch 172/500
4/4 [==============================] - 0s 16ms/step - loss: 0.0073 - accuracy: 1.0000
Epoch 173/500
4/4 [==============================] - 0s 13ms/step - loss: 0.0073 - accuracy: 1.0000
Epoch 174/500
4/4 [==============================] - 0s 11ms/step - loss: 0.0068 - accuracy: 1.0000
Epoch 175/500


4/4 [==============================] - 0s 10ms/step - loss: 1.0344e-04 - accuracy: 1.0000
Epoch 246/500
4/4 [==============================] - 0s 10ms/step - loss: 1.2284e-04 - accuracy: 1.0000
Epoch 247/500
4/4 [==============================] - 0s 10ms/step - loss: 6.5405e-04 - accuracy: 1.0000
Epoch 248/500
4/4 [==============================] - 0s 11ms/step - loss: 0.0011 - accuracy: 1.0000
Epoch 249/500
4/4 [==============================] - 0s 10ms/step - loss: 7.3309e-05 - accuracy: 1.0000
Epoch 250/500
4/4 [==============================] - 0s 10ms/step - loss: 6.7137e-05 - accuracy: 1.0000
Epoch 251/500
4/4 [==============================] - 0s 10ms/step - loss: 6.3101e-05 - accuracy: 1.0000
Epoch 252/500
4/4 [==============================] - 0s 10ms/step - loss: 5.9838e-05 - accuracy: 1.0000
Epoch 253/500
4/4 [==============================] - 0s 10ms/step - loss: 5.7190e-05 - accuracy: 1.0000
Epoch 254/500
4/4 [==============================] - 0s 11ms/step - loss: 5.3909e-

4/4 [==============================] - 0s 13ms/step - loss: 3.0333e-06 - accuracy: 1.0000
Epoch 325/500
4/4 [==============================] - 0s 10ms/step - loss: 2.9022e-06 - accuracy: 1.0000
Epoch 326/500
4/4 [==============================] - 0s 11ms/step - loss: 2.8466e-06 - accuracy: 1.0000
Epoch 327/500
4/4 [==============================] - 0s 10ms/step - loss: 2.7581e-06 - accuracy: 1.0000
Epoch 328/500
4/4 [==============================] - 0s 10ms/step - loss: 2.7389e-06 - accuracy: 1.0000
Epoch 329/500
4/4 [==============================] - 0s 11ms/step - loss: 2.6360e-06 - accuracy: 1.0000
Epoch 330/500
4/4 [==============================] - 0s 10ms/step - loss: 2.7671e-06 - accuracy: 1.0000
Epoch 331/500
4/4 [==============================] - 0s 10ms/step - loss: 2.4759e-06 - accuracy: 1.0000
Epoch 332/500
4/4 [==============================] - 0s 9ms/step - loss: 2.3719e-06 - accuracy: 1.0000
Epoch 333/500
4/4 [==============================] - 0s 9ms/step - loss: 2.4022

4/4 [==============================] - 0s 10ms/step - loss: 7.3368e-07 - accuracy: 1.0000
Epoch 404/500
4/4 [==============================] - 0s 10ms/step - loss: 7.2681e-07 - accuracy: 1.0000
Epoch 405/500
4/4 [==============================] - 0s 10ms/step - loss: 7.1562e-07 - accuracy: 1.0000
Epoch 406/500
4/4 [==============================] - 0s 11ms/step - loss: 7.0297e-07 - accuracy: 1.0000
Epoch 407/500
4/4 [==============================] - 0s 10ms/step - loss: 7.0586e-07 - accuracy: 1.0000
Epoch 408/500
4/4 [==============================] - 0s 10ms/step - loss: 6.9358e-07 - accuracy: 1.0000
Epoch 409/500
4/4 [==============================] - 0s 10ms/step - loss: 6.9214e-07 - accuracy: 1.0000
Epoch 410/500
4/4 [==============================] - 0s 10ms/step - loss: 6.7877e-07 - accuracy: 1.0000
Epoch 411/500
4/4 [==============================] - 0s 12ms/step - loss: 6.7985e-07 - accuracy: 1.0000
Epoch 412/500
4/4 [==============================] - 0s 10ms/step - loss: 6.75

4/4 [==============================] - 0s 11ms/step - loss: 4.0423e-07 - accuracy: 1.0000
Epoch 482/500
4/4 [==============================] - 0s 11ms/step - loss: 4.0314e-07 - accuracy: 1.0000
Epoch 483/500
4/4 [==============================] - 0s 11ms/step - loss: 4.0459e-07 - accuracy: 1.0000
Epoch 484/500
4/4 [==============================] - 0s 11ms/step - loss: 3.9953e-07 - accuracy: 1.0000
Epoch 485/500
4/4 [==============================] - 0s 10ms/step - loss: 3.9664e-07 - accuracy: 1.0000
Epoch 486/500
4/4 [==============================] - 0s 11ms/step - loss: 3.9339e-07 - accuracy: 1.0000
Epoch 487/500
4/4 [==============================] - 0s 10ms/step - loss: 3.9014e-07 - accuracy: 1.0000
Epoch 488/500
4/4 [==============================] - 0s 11ms/step - loss: 3.8653e-07 - accuracy: 1.0000
Epoch 489/500
4/4 [==============================] - 0s 11ms/step - loss: 3.8580e-07 - accuracy: 1.0000
Epoch 490/500
4/4 [==============================] - 0s 10ms/step - loss: 3.84

In [83]:
print("X_enc.shape:", X_enc.shape)
print("X_dec.shape:", X_dec.shape)
print("Y_dec.shape:", Y_dec.shape)


X_enc.shape: (2, 4, 172)
X_dec.shape: (110, 3, 172)
Y_dec.shape: (110, 3, 1)


In [89]:
# 쉬운문제
easy_test=[['cold', 'PP'],
           ['fact', 'PP'],
           ['love', 'PP'],
           ['luck', 'PP'],
           ['milk', 'PP']]
enc_in, dec_in, _ = encoding(easy_test)
enc_in = np.array(enc_in)
dec_in = np.array(dec_in)
enc_in.shape, dec_in.shape

((5, 4, 172), (5, 3, 172))

In [90]:
# 위의 문제 예측

# 위의 문제 예측하기
pred = model.predict([enc_in, dec_in])
pred.argmax(axis=-1)

1/1 [==============================] - 1s 638ms/step


array([[ 33,  47,   1],
       [ 97, 110,   1],
       [ 97,  67,   1],
       [166, 126,   1],
       [125, 129,   1]], dtype=int64)

In [91]:
char_to_num['감'],alpha[32]

(33, '간')

In [94]:
for test, yhat in zip(easy_test, pred):
#     print(test[0], yhat.argmax(axis=-1))
    eng = test[0]
    hat = np.argmax(yhat, axis=-1)
    kor = ''.join([alpha[h] for h in hat[:-1]])
    print("{} => {}".format(eng, kor))

cold => 감기
fact => 사실
love => 사랑
luck => 행운
milk => 우유


In [95]:
# 어려운 문제
hard_test=[['lvoe', 'PP'],
           ['loev', 'PP'],
           ['love', 'PP'],
           ['olve', 'PP'],
           ['eovl', 'PP']]
enc_in, dec_in, _ = encoding(hard_test)
enc_in = np.array(enc_in)
dec_in = np.array(dec_in)